### Mapping MAF to histologic pattern per tile and wsi classificaion in csv format

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:

# ---------------- CONFIG ----------------

CSV_DIR = Path("/home/rapids/notebooks/slima/outputs/inference_results_parallel2")
MAF_PATH = Path("/home/rapids/notebooks/slima/MAF/cohortMAF.2025-11-16.maf")

# Output files
OUT_SUMMARY_CSV = CSV_DIR / "tcga_histologic_pattern_summary_per_slide_dec2025.csv"
OUT_MERGED_MAF = MAF_PATH.with_name(MAF_PATH.stem + "_with_histologic_patterns_dec2025.maf")

# Mapping from class id -> histologic pattern
ID2LABEL = {
    0: "acinar",
    1: "lepidic",
    2: "micropapillary",
    3: "mucinous",
    4: "papillary",
    5: "solid",
}

# Column in the MAF that holds the full TCGA sample barcode
MAF_BARCODE_COL = "Tumor_Sample_Barcode"   # adjust if your MAF uses another name



In [3]:

# ------------- HELPERS -------------------

def extract_slide_id_from_csv_name(csv_path: Path) -> str:
    """
    From a filename like:
      TCGA-55-8094-01Z-00-DX1.8dc29615-e124-4f17-81a1-c0b20c38d12c_tiles_384_predictions.csv
    return:
      TCGA-55-8094-01Z-00-DX1.8dc29615-e124-4f17-81a1-c0b20c38d12c
    """
    stem = csv_path.stem
    return stem.split("_tiles_")[0]


def slide_id_to_case_barcode(slide_id: str) -> str:
    """
    Map WSI slide id to TCGA case barcode.

    Example:
      slide_id = "TCGA-55-8094-01Z-00-DX1.8dc2..."
      -> "TCGA-55-8094"

    If you want more granularity (e.g. sample-level), change the join length.
    """
    parts = slide_id.split("-")
    # patient/case barcode = first 3 fields: TCGA-XX-XXXX
    return "-".join(parts[:3])


def maf_barcode_to_case_barcode(barcode: str) -> str:
    """
    Map MAF Tumor_Sample_Barcode to TCGA case barcode.
    E.g. "TCGA-55-8094-01A-01D-XXXX-XX" -> "TCGA-55-8094"
    """
    if pd.isna(barcode):
        return np.nan
    parts = str(barcode).split("-")
    return "-".join(parts[:3])


# ------------- STEP 1: aggregate CSVs ---------------

def build_slide_histology_summary(csv_dir: Path) -> pd.DataFrame:
    csv_files = sorted(csv_dir.glob("*_tiles_*_predictions.csv"))
    print(f"[INFO] Found {len(csv_files)} prediction CSV files in {csv_dir}")

    records = []

    for csv_path in csv_files:
        slide_id = extract_slide_id_from_csv_name(csv_path)
        case_barcode = slide_id_to_case_barcode(slide_id)

        df = pd.read_csv(csv_path)

        if "pred_class" not in df.columns:
            print(f"[WARN] CSV {csv_path} has no 'pred_class' column, skipping.")
            continue

        n_tiles = len(df)
        if n_tiles == 0:
            print(f"[WARN] CSV {csv_path} is empty, skipping.")
            continue

        # Count tiles per class
        counts = df["pred_class"].value_counts().to_dict()

        # Build a single row with counts and percentages
        row = {
            "slide_id": slide_id,
            "case_barcode": case_barcode,
            "n_tiles_total": n_tiles,
        }

        # Initialize all to 0
        for cid, label in ID2LABEL.items():
            row[f"n_{label}"] = 0
            row[f"pct_{label}"] = 0.0

        for cid, n in counts.items():
            cid_int = int(cid)
            label = ID2LABEL.get(cid_int, f"class_{cid_int}")
            row[f"n_{label}"] = n
            row[f"pct_{label}"] = n / n_tiles

        records.append(row)

    summary_df = pd.DataFrame.from_records(records)
    if not summary_df.empty:
        summary_df.to_csv(OUT_SUMMARY_CSV, index=False)
        print(f"[OK] Wrote slide-level histologic summary to {OUT_SUMMARY_CSV}")
    else:
        print("[WARN] No summary rows generated; check CSV directory / patterns.")

    return summary_df


# ------------- STEP 2: merge with MAF ---------------

def merge_summary_into_maf(summary_df: pd.DataFrame, maf_path: Path) -> pd.DataFrame:
    print(f"[INFO] Loading MAF from {maf_path}")
    maf_df = pd.read_csv(maf_path, sep="\t", comment="#", dtype=str)

    if MAF_BARCODE_COL not in maf_df.columns:
        raise ValueError(
            f"MAF file {maf_path} has no column '{MAF_BARCODE_COL}'. "
            f"Available columns: {maf_df.columns.tolist()}"
        )

    # Derive case barcode for each row in MAF
    maf_df["case_barcode"] = maf_df[MAF_BARCODE_COL].apply(maf_barcode_to_case_barcode)

    # A single case can have multiple MAF rows; we want per-case histology.
    # summary_df is already per (slide_id, case_barcode). If multiple slides
    # exist per case, we can aggregate them here; for now we choose maximum
    # percentage per pattern across slides for that case.
    if summary_df.empty:
        print("[WARN] Summary DF is empty; returning original MAF.")
        return maf_df

    agg_cols = [c for c in summary_df.columns if c.startswith("pct_") or c.startswith("n_")]
    case_histology = (
        summary_df
        .groupby("case_barcode")[agg_cols]
        .max()  # or .mean(), depending on what you want
        .reset_index()
    )

    print(f"[INFO] Aggregated histology for {len(case_histology)} unique cases")

    merged = maf_df.merge(case_histology, on="case_barcode", how="left")

    merged.to_csv(OUT_MERGED_MAF, sep="\t", index=False)
    print(f"[OK] Wrote merged MAF with histologic patterns to {OUT_MERGED_MAF}")

    return merged



In [4]:

def main():
    summary_df = build_slide_histology_summary(CSV_DIR)
    merge_summary_into_maf(summary_df, MAF_PATH)


In [5]:


if __name__ == "__main__":
    main()


[INFO] Found 763 prediction CSV files in /home/rapids/notebooks/slima/outputs/inference_results_parallel2
[OK] Wrote slide-level histologic summary to /home/rapids/notebooks/slima/outputs/inference_results_parallel2/tcga_histologic_pattern_summary_per_slide_dec2025.csv
[INFO] Loading MAF from /home/rapids/notebooks/slima/MAF/cohortMAF.2025-11-16.maf
[INFO] Aggregated histology for 703 unique cases
[OK] Wrote merged MAF with histologic patterns to /home/rapids/notebooks/slima/MAF/cohortMAF.2025-11-16_with_histologic_patterns_dec2025.maf


In [6]:

# Input: slide-level summary we already created
SLIDE_SUMMARY = Path(
    "/home/rapids/notebooks/slima/outputs/inference_results_parallel2/"
    "tcga_histologic_pattern_summary_per_slide_dec2025.csv"
)

# Output: one row per TCGA case
OUT_CASE_CSV = Path(
    "/home/rapids/notebooks/slima/outputs/inference_results_parallel2/"
    "tcga_case_histologic_patterns_dec2025.csv"
)


In [7]:

df = pd.read_csv(SLIDE_SUMMARY)

# All histology columns
hist_cols = [c for c in df.columns if c.startswith("n_") or c.startswith("pct_")]

# Aggregate per case_barcode (you can switch .max() to .mean() or .sum() if you prefer)
case_df = (
    df.groupby("case_barcode")[hist_cols]
      .max()
      .reset_index()
)


In [8]:

case_df.to_csv(OUT_CASE_CSV, index=False)
print(f"Case-level histologic CSV written to: {OUT_CASE_CSV}")


Case-level histologic CSV written to: /home/rapids/notebooks/slima/outputs/inference_results_parallel2/tcga_case_histologic_patterns_dec2025.csv


In [9]:

MAF_WITH_HIST = Path(
    "/home/rapids/notebooks/slima/MAF/cohortMAF.2025-11-16_with_histologic_patterns_dec2025.maf"
)
OUT_MAF_CSV = Path(
    "/home/rapids/notebooks/slima/MAF/cohortMAF.2025-12-17_with_histologic_patterns_dec2025.csv"
)


In [10]:

df = pd.read_csv(MAF_WITH_HIST, sep="\t", dtype=str)


In [11]:
df.to_csv(OUT_MAF_CSV, index=False)
print(f"Full MAF+histology CSV written to: {OUT_MAF_CSV}")


Full MAF+histology CSV written to: /home/rapids/notebooks/slima/MAF/cohortMAF.2025-12-17_with_histologic_patterns_dec2025.csv
